# Assignment APIs tutorial

In this notebook we are using the python package "millionaire-client" to interact with the deployed application for the NLP assignment 2026.

Required files:
- Directory called "millionaire_client"
- 1 colab notebook

Both files must be saved in a directory in your Google Drive, for example:
```
gDrive_home/
├── Colab Notebooks/
│   └── NLP_assignment/
│       ├── PoliMillionaire.ipynb <-- Your notebook
│       └── millionaire_client/ <-- Directory provided
```

### Sign up procedure
Before showing you how the api work, you need to signup from a web browser.
- Paste this link into your browser [http://131.175.15.22:51111/](http://131.175.15.22:51111/) this is where the demo is deployed
- You will see a standard login/sign up screen, please click on sign up
- In the "email" field please enter your politecnico email, you are allwed to create only 1 account using the same email you registered to the NLP course
- Choose whever username/password you prefer (be creative ;))

### Game interaction

Once you signed up, you can start interacting already from the api.

First of all, let's connect your drive to this Colab Notebook

In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

Then we need to add our python package "millionaire_client" to the system path, so python can see it.

In [ ]:
import sys
import os

# define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment'

# append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# verify the path was added
print(sys.path)

Let's import the client classes

In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

You can save your password in a Colab secret (the "key" icon on the tab on the left) and import it into your notebook.

In [ ]:
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

Now keep the API_URL as stated, but please change the username and password to be the ones you used during sign up session.

In [ ]:
API_URL = "http://131.175.15.22:51111/"
username = "nicolo"
password = pwd

Now we can instantiate a MillionaireClient object and call the login method, which takes as parameters username and password.

In [ ]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nwelcome, {user.username}! (role: {user.role})")
except AuthenticationError as e:
    print(f"login failed: {e}")

After login, the web page is showing you different types of competitions, for each of them you can choose to play a game or to see the leaderboard. For now let's list all of the.

In [ ]:
# list available competitions
print("\n=== available competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")

In [ ]:
# choose a competition id
comp_id = 1

After choosing a competition, we can start a game! We can choose to start a game by calling `game = client.game.start(competition_id=comp_id)`. The object game is the one that is handling the game itself, we can call:
- game.current_question.text : to know the current question in text format
- game.current_level: to check the current level of difficulty of the question
- game.current_question.options: to check the possible choices we have to answer the question
- game.answer: to send to the server the answer we choose (the integer corresponding to our choice) and get the response (either correct or incorrect)

WATCH OUT! Each question has a timer, you have maximum 30 seconds to answer the question. As of now, if you exceed the maximum allowed time, there is not a "push notification". You still have to submit your answer anyway and, even though the answer was correct, you will get a TimedOut response!

In [ ]:
def play_game(game):
  # play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("no question available. game may have ended.")
          break

      print(f"\n--- level {game.current_level} ---")
      print(f"q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      # get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\ntime remaining: {time_left:.1f}s")

      # get answer
      try:
          answer_input = input("\nYour answer (option ID): ").strip()
          answer_id = int(answer_input)
      except ValueError:
          print("invalid input. please enter a number.")
          continue

      # submit answer
      result = game.answer(answer_id)

      if result.correct:
          print(" correct!")
          if result.game_over:
              print(f"\n congratulations! you completed the game!")
              print(f" final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("timed out!")
        print(f"\n game over!")
        print(f" final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" wrong answer!")
          print(f"\n game over!")
          print(f" final earnings: ${result.earned_amount:,.2f}")

  print("\n=== game summary ===")
  print(f"reached level: {game.current_level}")
  print(f"total earnings: ${game.earned_amount:,.2f}")

In [ ]:
# start the game
print("\n=== starting game ===")
game = client.game.start(competition_id=comp_id)
print(f"session id: {game.session_id}")
print(f"total number of questions: {game.state.competition.max_levels}")
print()
play_game(game)

In [ ]:
# show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (level {entry.reached_level}){marker}")